# Holevo information for the symmetric-subspace probe state ($d \ge 3$)

Numerics for the section **"Optimization of the Holevo information"** of
*Quantum Programmable Reflections* (Schoute, Grinko, Subaşı, Volkoff).

When the probe weights are concentrated on the symmetric irrep $\theta_n$
($q_\lambda = \delta_{\lambda,\theta_n}$), the vector $|v_\nu\rangle$ reduces to a single scalar
$v_{\nu_k}$ built from the *reduced* $U(d)$ Clebsch–Gordan coefficients

$$RC^{(\vec 0_{d-1},-n),(n,\vec 0_{d-1}),\,\nu_k}_{(\vec 0_{d-1},-q),(q,\vec 0_{d-1}),\,\vec 0_{d-1}},$$

which have the explicit combinatorial form of Vilenkin & Klimyk, *Representation of Lie Groups
and Special Functions*, Vol. 3, §18.2.6 Eq. (6).  Using it, the Holevo information
$S(\widetilde{\mathcal T(\rho)})$ can be computed for much larger $n$ and $d$ than the general
optimization in `lower_bound_optimization.ipynb`.  The results for $d = 3,\dots,20$ and
$n = 1,\dots,40$ are cached in `holevo_information_data_d_3_20_n_1_41.pkl` and plotted in
`log_log_holevo_information_plot.ipynb`.

In [ ]:
# Activate the pinned Julia environment (Project.toml), searching this directory and its parents.
import Pkg
function find_project(dir = pwd())
    while !isfile(joinpath(dir, "Project.toml"))
        parent = dirname(dir)
        parent == dir && error("Could not find Project.toml -- run this notebook from the repository directory.")
        dir = parent
    end
    return dir
end
Pkg.activate(find_project())
# Pkg.instantiate()   # uncomment on first run to install the pinned dependencies

In [ ]:
using LinearAlgebra
using SpecialFunctions   # lgamma, used in the combinatorial CG-coefficient formula
using Plots

## The closed-form Holevo information

`S(λ, μ)` is the logarithm of the function whose $\sqrt{\exp(\cdot)}$ gives Eq. (3) on p. 373 of
Vilenkin & Klimyk, Vol. 3 (valid for $|\lambda| \ge |\mu|$).  `ScalarFactor(n, q, k, d)` is the
reduced CG coefficient above, and `symmetric_subspace_entropy(n, d)` assembles
$S(\widetilde{\mathcal T(\rho)}) = -\sum_k d_{\nu_k}\,\lVert v_{\nu_k}\rVert^2 \log_2 \lVert v_{\nu_k}\rVert^2$.

In [ ]:
function S(λ, μ)
    @assert length(λ) >= length(μ)
    lλ = [λ[j] - j for j in eachindex(λ)]
    lμ = [μ[j] - j for j in eachindex(μ)]
    pos = sum(lgamma(lλ[i] - lμ[j] + 1)     for j in eachindex(μ) for i in 1:j)
    neg = sum(lgamma(lμ[i] - lλ[j] - 1 + 1) for j in 2:length(λ) for i in 1:j-1)
    return pos - neg
end

# Weyl dimension formula for the SU(p) (equivalently GL(p)) irrep with highest weight λ.
function WeylDimensionFormula(λ)
    l = [λ[j] - j for j in eachindex(λ)]
    return prod((l[i] - l[j]) / (j - i) for j in eachindex(λ) for i in 1:j-1)
end

# Reduced U(d) CG coefficient  RC^{(0_{d-1},-n),(n,0_{d-1}),ν_k}_{(0_{d-1},-q),(q,0_{d-1}),0_{d-1}}
# from Vilenkin & Klimyk, Vol. 3, §18.2.6 Eq. (6).
function ScalarFactor(n, q, k, d)
    m_d    = Tuple(vcat(fill(0, d - 1), -n))
    m_dm1  = Tuple(vcat(fill(0, d - 2), -q))
    mp_d   = Tuple(vcat(k, fill(0, d - 2), -k))
    mp_dm1 = Tuple(fill(0, d - 1))

    prefactor = sqrt(exp(
        S(mp_d, mp_d) + S(m_d, m_dm1) + S(mp_dm1, mp_dm1) + S(m_dm1, m_dm1)
        - S(mp_d, m_d) - S(mp_d, mp_dm1)))

    alternating_sum = sum(0:q; init = big(0)) do l
        r = Tuple(vcat(fill(0, d - 2), -q + l))
        (-1)^l * exp(S(mp_d, r) + S(r, r) - S(m_d, r) - S(mp_dm1, r) - S(r, m_dm1))
    end

    return sqrt(factorial(big(n - q))) * prefactor * alternating_sum
end

# Holevo information S(𝒯̃(ρ)) for the probe state with weights q_λ = δ_{λ,θ_n}  (d ≥ 3).
function symmetric_subspace_entropy(n, d)
    θ_n = Tuple(vcat(n, fill(0, d - 1)))            # the symmetric irrep
    d_θ = WeylDimensionFormula(θ_n)
    S_HI = 0.0
    for k in 0:n
        ν_k  = Tuple(vcat(k, fill(0, d - 2), -k))
        v_νk = sum(ScalarFactor(n, q, k, d) * sqrt(binomial(q + d - 2, d - 2)) for q in 0:n)
        p_νk = v_νk^2 / d_θ                          # = d_{ν_k} ‖v_{ν_k}‖²
        S_HI += -p_νk * log2(p_νk / WeylDimensionFormula(ν_k))
    end
    return S_HI
end

## Sanity checks

In [ ]:
# exp(S(λ, λ)) should equal WeylDimensionFormula(λ) · ∏_{k=1}^{p-1} k!
n, d = 3, 5
λ = Tuple(vcat(n, fill(0, d - 1)))
@show round(exp(S(λ, λ)))
@show WeylDimensionFormula(λ) * prod(factorial(k) for k in 1:length(λ) - 1)

In [ ]:
# Inspect the matrix of reduced CG coefficients and the (pseudo)inverse of RC · RCᵀ.
n, d = 6, 3
RC = Float64[ScalarFactor(n, q, k, d) for q in 0:n, k in 0:n]
display(RC)
display(Diagonal(pinv(round.(RC * transpose(RC); digits = 12))))

## Holevo information vs. the upper bound

Pick the dimensions and program sizes to evaluate.  To regenerate
`holevo_information_data_d_3_20_n_1_41.pkl`, use `dims = 3:20`, `copies = 1:40`
(this is slow for the larger values).

In [ ]:
dims   = 10:10
copies = 1:22

holevo = [[symmetric_subspace_entropy(n, d)     for n in copies] for d in dims]
ub     = [[2 * log2(binomial(n + d - 1, d - 1)) for n in copies] for d in dims]

In [ ]:
plt = plot(title = "Holevo information", xlabel = "n", legend = :topleft)
for (j, d) in enumerate(dims)
    plot!(plt, copies, holevo[j], label = "d = $d (computed)",     markershape = :circle)
    plot!(plt, copies, ub[j],     label = "d = $d (upper bound)",  markershape = :ltriangle)
end
plt

In [ ]:
# Gap to the upper bound on a log–log scale:  1 - r ~ c(d) / n^α  with α ≈ 0.3.
# (The first few n are dropped because the bound is (near) saturated there, e.g. exactly at n = 1 for d = 3.)
plt = plot(title = "gap 1 − (Holevo information / upper bound)", xlabel = "log n", ylabel = "log(1 − r)")
for (j, d) in enumerate(dims)
    idx = 4:length(copies)
    plot!(plt, log.(copies[idx]), log.(1 .- holevo[j][idx] ./ ub[j][idx]), label = "d = $d", markershape = :circle)
end
plt